# 01 — Data Ingestion / ดึงข้อมูล

**EN.** Wraps `scripts/ingest_all.py` in Colab. Pulls a multi-year window into the Drive-backed parquet store at `data/raw/station_id={id}/date={iso}/obs.parquet`. Idempotent — partitions that already exist are skipped unless `--force`.

**TH.** ห่อ `scripts/ingest_all.py` สำหรับ Colab. ดึงข้อมูลย้อนหลังหลายปีลง parquet store บน Drive ที่ `data/raw/station_id={id}/date={iso}/obs.parquet`. รันซ้ำได้ — partition ที่มีอยู่จะถูกข้ามเว้นแต่ใช้ `--force`.

**Recommended data window / ช่วงข้อมูลที่แนะนำ**

- NASA POWER (no auth) — last **5 years** for solar + cloud + sfc temp.
- ERA5 via CDS (`CDSAPI_KEY`) — last **3 years** at 0.25° resolution.
- TMD (`TMD_API_KEY`) — current Thai station data, last **1–2 years** depending on availability.
- Loader prefers ERA5 over NASA POWER for the same `(station_id, ts_utc)` (see `app/data/loaders.py`).


In [ ]:
# --- bootstrap (re-run if you opened this notebook before 00_setup) -------
import os, sys
REPO_DIR = "/content/Heat-wave-backend"
if not os.path.exists(REPO_DIR):
    !bash {REPO_DIR}/scripts/colab_bootstrap.sh || true
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(os.getcwd())


In [ ]:
# --- write CDS credentials from Colab Secrets if present ------------------
import os, pathlib
cds_key = os.environ.get("CDSAPI_KEY")
if cds_key:
    cdsapirc = pathlib.Path("/root/.cdsapirc")
    cdsapirc.write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {cds_key}\n"
    )
    print(f"wrote {cdsapirc}")
else:
    print("CDSAPI_KEY not set — ERA5 ingest will be skipped.")


In [ ]:
# --- run NASA POWER ingest (no auth, 5y window) ---------------------------
from datetime import date, timedelta
end = date.today() - timedelta(days=1)
start_5y = end - timedelta(days=5 * 365)
print(f"NASA POWER window: {start_5y} -> {end}")
!python scripts/ingest_all.py --sources nasa_power --start {start_5y} --end {end}


In [ ]:
# --- run ERA5 ingest (3y window, requires CDSAPI_KEY) --------------------
import os
from datetime import date, timedelta
if os.environ.get("CDSAPI_KEY"):
    end = date.today() - timedelta(days=1)
    start_3y = end - timedelta(days=3 * 365)
    print(f"ERA5 window: {start_3y} -> {end}")
    !python scripts/ingest_all.py --sources era5 --start {start_3y} --end {end}
else:
    print("Skipping ERA5 — CDSAPI_KEY not set.")


In [ ]:
# --- (optional) TMD ingest, recent window only ----------------------------
import os
from datetime import date, timedelta
if os.environ.get("TMD_API_KEY"):
    end = date.today() - timedelta(days=1)
    start_2y = end - timedelta(days=2 * 365)
    print(f"TMD window: {start_2y} -> {end}")
    !python scripts/ingest_all.py --sources tmd --start {start_2y} --end {end}
else:
    print("Skipping TMD — TMD_API_KEY not set.")


In [ ]:
# --- coverage summary across all stations ---------------------------------
from datetime import date, timedelta
from app.data.stations import STATIONS
from app.data.loaders import read_observations

end = date.today() - timedelta(days=1)
start = end - timedelta(days=5 * 365)

for sid in STATIONS:
    try:
        df = read_observations(sid, start, end)
        n = len(df)
        srcs = df["source"].value_counts().to_dict() if "source" in df.columns else {}
        print(f"  {sid}: {n:>7d} rows | sources={srcs}")
    except Exception as exc:
        print(f"  {sid}: ERROR {exc}")


## Next steps / ขั้นถัดไป

**EN.** When coverage looks healthy, jump to `02_train_*.ipynb` (per backend) to train v3 forecasters. Artifacts will land in `app/models/forecast_v3/{station_id}/h{H}/` (Drive-backed) and can be pulled back to the local repo via the workflow described in `docs/colab-training.md`.

**TH.** เมื่อ coverage ดูครบถ้วนแล้ว ไปที่ `02_train_*.ipynb` เพื่อเทรน v3 forecaster. ผลลัพธ์จะถูกเก็บที่ `app/models/forecast_v3/{station_id}/h{H}/` (อยู่บน Drive) และสามารถ push กลับเข้า repo ตามขั้นตอนใน `docs/colab-training.md`.
